# Phase 1: stack validation on AirfRANS

Goal: train a small Transolver on the AirfRANS `scarce` task and confirm that
the vendored Physics-Attention block produces a sensible flow field on a
held-out airfoil. The acceptance figure is a two-panel plot (surface Cp + |U|
on the mesh) for one held-out case. The acceptance bar is per-channel
relative L2 within roughly 2x the published AirfRANS Transolver number.

Intended runtime is one Colab T4 session. Train at batch_size=1 with 32k
random subsampled points per epoch.

Configuration: 4 layers, d_model=128, slice_num=32, mlp_ratio=2,
unified_pos with the AirfRANS bounds [-2, 4] x [-1.5, 1.5].

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle/working").exists()
REPO_URL = os.environ.get("TRANSOLVER_REPO_URL",
    "https://github.com/zeteixeira03/transolver-hypersonic.git")

def _locate_repo() -> Path:
    here = Path.cwd().resolve()
    if (here / "src").exists():
        return here
    if (here.parent / "src").exists():
        return here.parent

    if IN_COLAB:
        repo_dir = Path("/content/transolver-hypersonic")
    elif IN_KAGGLE:
        repo_dir = Path("/kaggle/working/transolver-hypersonic")
    else:
        raise RuntimeError("cannot locate src/; run from the repo or set TRANSOLVER_REPO_URL")

    if not repo_dir.exists():
        print(f"cloning {REPO_URL} into {repo_dir} ...")
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)])
    else:
        # pull latest on existing checkout (Kaggle persists /kaggle/working between commits)
        try:
            subprocess.check_call(["git", "-C", str(repo_dir), "pull", "--ff-only"])
        except subprocess.CalledProcessError:
            pass
    return repo_dir

PROJECT_ROOT = _locate_repo()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("project root:", PROJECT_ROOT)


In [ ]:
%pip install --quiet 'airfrans>=0.1.5' 'einops>=0.7' 'tqdm>=4.66'


In [ ]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(0))

## 2. Download AirfRANS

In [ ]:
import os, glob, zipfile
from pathlib import Path
import airfrans as af

# AirfRANS data resolution, in priority order:
# 1. AIRFRANS_ROOT env var pointing at an extracted Dataset folder
#    (manifest.json inside).
# 2. Kaggle: any attached dataset under /kaggle/input/*/Dataset/manifest.json.
# 3. Colab with AIRFRANS_DRIVE_ZIP pointing at a Drive-hosted Dataset.zip:
#    mount Drive and unzip into /content/airfrans/Dataset.
# 4. Fall back to downloading the dataset fresh (~9 GB).

def _resolve_data_root() -> Path:
    env_root = os.environ.get("AIRFRANS_ROOT")
    if env_root and (Path(env_root) / "manifest.json").exists():
        return Path(env_root)

    if IN_KAGGLE:
        hits = glob.glob("/kaggle/input/*/Dataset/manifest.json") \
             + glob.glob("/kaggle/input/*/manifest.json")
        if hits:
            return Path(hits[0]).parent

    if IN_COLAB:
        fresh_target = Path("/content/airfrans/Dataset")
    elif IN_KAGGLE:
        fresh_target = Path("/kaggle/working/airfrans/Dataset")
    else:
        fresh_target = PROJECT_ROOT / "data" / "airfrans" / "Dataset"

    drive_zip_env = os.environ.get("AIRFRANS_DRIVE_ZIP")
    if IN_COLAB and drive_zip_env:
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
        zip_path = Path(drive_zip_env)
        if zip_path.exists() and not (fresh_target / "manifest.json").exists():
            fresh_target.parent.mkdir(parents=True, exist_ok=True)
            print(f"unzipping {zip_path} into {fresh_target.parent} ...")
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(fresh_target.parent)
            return fresh_target

    if not (fresh_target / "manifest.json").exists():
        fresh_target.parent.mkdir(parents=True, exist_ok=True)
        print(f"downloading AirfRANS to {fresh_target.parent} (~9 GB)")
        af.dataset.download(root=str(fresh_target.parent), unzip=True)
    return fresh_target

DATA_ROOT = _resolve_data_root()
print("AirfRANS root:", DATA_ROOT)
assert (DATA_ROOT / "manifest.json").exists(), "manifest.json not found at DATA_ROOT"


## 3. Datasets and norm stats

In [ ]:
from src.data.airfrans import (
    AirfRANSDataset,
    compute_norm_stats,
    load_split,
)

TASK = 'scarce'
SUBSAMPLE = 32_000

# load train, compute streaming stats, then load test. Avoids peaking
# with both splits live during normalization. The earlier version
# concatenated everything and OOM'd Colab's 12 GB.
train_arrs, train_names = load_split(DATA_ROOT, task=TASK, train=True)
print(f'train: {len(train_arrs)} cases, first shape: {train_arrs[0].shape}')

stats = compute_norm_stats(train_arrs)
print('x_mean', stats.x_mean.numpy().round(3))
print('x_std ', stats.x_std.numpy().round(3))
print('y_mean', stats.y_mean.numpy().round(3))
print('y_std ', stats.y_std.numpy().round(3))

train_ds = AirfRANSDataset(train_arrs, train_names, stats, subsample=SUBSAMPLE)

test_arrs, test_names = load_split(DATA_ROOT, task=TASK, train=False)
print(f'test: {len(test_arrs)} cases')
test_ds = AirfRANSDataset(test_arrs, test_names, stats, subsample=None)

## 4. Model + optimizer

In [ ]:
from src.models.transolver import Transolver

model = Transolver(
    space_dim=7,
    fun_dim=0,
    out_dim=4,
    n_hidden=128,
    n_layers=4,
    n_head=8,
    mlp_ratio=2,
    slice_num=32,
    dropout=0.0,
    unified_pos=True,
    grid_ref=8,
    grid_bounds=(-2.0, 4.0, -1.5, 1.5),
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f'parameters: {n_params/1e6:.2f}M')

In [ ]:
from torch.utils.data import DataLoader
from src.training.loop import TrainConfig, collate_single, train_one_epoch, evaluate

cfg = TrainConfig(
    epochs=200,
    lr=1e-3,
    weight_decay=0.0,
    surface_weight=1.0,
    batch_size=1,
    device=DEVICE,
    val_every=20,
    amp=(DEVICE == 'cuda'),
)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)
scaler = torch.cuda.amp.GradScaler(enabled=cfg.amp)
loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=collate_single)


## 5. Train

In [ ]:
import time
from tqdm.auto import tqdm

history = []
pbar = tqdm(range(cfg.epochs))
t_first = None
for epoch in pbar:
    t0 = time.time()
    train_metrics = train_one_epoch(model, loader, optimizer, cfg, scaler=scaler)
    scheduler.step()
    dt = time.time() - t0
    row = {'epoch': epoch, 'sec': dt, **train_metrics}

    if epoch % cfg.val_every == cfg.val_every - 1 or epoch == cfg.epochs - 1:
        val_metrics = evaluate(model, test_ds, stats, DEVICE)
        row.update(val_metrics)

    history.append(row)
    pbar.set_postfix({k: f'{v:.3f}' for k, v in row.items() if isinstance(v, (int, float))})

    if epoch == 0:
        t_first = dt
        eta_min = dt * cfg.epochs / 60
        print(f'first epoch {dt:.1f}s -> projected total ~{eta_min:.1f} min for {cfg.epochs} epochs')
        if eta_min > 600:
            print('WARNING: projected runtime exceeds a 10h Colab session. Consider reducing epochs.')

print('final:', history[-1])


## 6. Acceptance figure

In [ ]:
from src.eval.plots import plot_acceptance

samples_dir = PROJECT_ROOT / 'data' / 'samples'
samples_dir.mkdir(parents=True, exist_ok=True)
fig_path = samples_dir / 'phase1_acceptance.png'

fig, metrics = plot_acceptance(
    model=model,
    dataset=test_ds,
    stats=stats,
    device=DEVICE,
    sim_index=0,
    save_path=fig_path,
)
print('saved:', fig_path)
print('per-channel rel-L2:', metrics)

In [ ]:
ckpt_dir = PROJECT_ROOT / 'checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)
torch.save({
    'state_dict': model.state_dict(),
    'stats': {
        'x_mean': stats.x_mean, 'x_std': stats.x_std,
        'y_mean': stats.y_mean, 'y_std': stats.y_std,
    },
    'history': history,
    'config': cfg.__dict__,
}, ckpt_dir / 'phase1_transolver_small.pt')
print('checkpoint saved')